# 1. Generate observations from the probabilistic grammar

This notebook generates **850 sentences** from the fictional probabilistic grammar.

The grammar is:

```text
S -> s M | i L

M -> F | r r F | r r r r F | l l F | l l r r F

F -> f e | p e | e

L -> l l | l l l l
```

The probabilities are the production probabilities discussed in class. Each complete sentence is generated by sampling the alternatives independently according to the probability attached to the selected nonterminal.

The result is saved as `observations.csv`, with one generated sentence per row.


In [1]:
import random
import pandas as pd

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
N_OBSERVATIONS = 850
OUTPUT_FILE = "observations.csv"

# Use an integer seed for reproducibility during demonstrations.
# Set to None if you want a different dataset on every execution.
SEED = 42

if SEED is not None:
    random.seed(SEED)


In [2]:
# ------------------------------------------------------------
# Probabilistic grammar
# ------------------------------------------------------------

GRAMMAR = {
    "S": [
        (0.90, ["s", "M"]),
        (0.10, ["i", "L"]),
    ],
    "M": [
        (0.20, ["F"]),
        (0.20, ["r", "r", "F"]),
        (0.20, ["r", "r", "r", "r", "F"]),
        (0.20, ["l", "l", "F"]),
        (0.20, ["l", "l", "r", "r", "F"]),
    ],
    "F": [
        (0.70, ["f", "e"]),
        (0.20, ["p", "e"]),
        (0.10, ["e"]),
    ],
    "L": [
        (0.50, ["l", "l"]),
        (0.50, ["l", "l", "l", "l"]),
    ],
}

def choose(options):
    """Sample one RHS according to its production probabilities."""
    probabilities = [p for p, _ in options]
    productions = [rhs for _, rhs in options]
    return random.choices(productions, weights=probabilities, k=1)[0]

def generate_sentence():
    """Generate one terminal sentence from S."""
    symbols = ["S"]

    while any(symbol in GRAMMAR for symbol in symbols):
        new_symbols = []
        for symbol in symbols:
            if symbol in GRAMMAR:
                new_symbols.extend(choose(GRAMMAR[symbol]))
            else:
                new_symbols.append(symbol)
        symbols = new_symbols

    return " ".join(symbols)


In [3]:
# ------------------------------------------------------------
# Generate the observations
# ------------------------------------------------------------

sentences = [generate_sentence() for _ in range(N_OBSERVATIONS)]

observations = pd.DataFrame({
    "id": range(1, N_OBSERVATIONS + 1),
    "sentence": sentences
})

observations.to_csv(OUTPUT_FILE, index=False)

print(f"Generated {len(observations)} observations.")
print(f"Saved to: {OUTPUT_FILE}")
observations.head(10)


Generated 850 observations.
Saved to: observations.csv


,id,sentence
0,1,s f e
1,2,s l l f e
2,3,s f e
3,4,s r r f e
4,5,s f e
5,6,s r r f e
6,7,s p e
7,8,s r r f e
8,9,i l l
9,10,s p e


## Check the empirical distribution

Because the observations are sampled randomly, the observed frequencies will generally be **close to**, but not exactly equal to, the theoretical probabilities.

For example, the theoretical probability of `s r r f e` is:

\[
0.90 \times 0.20 \times 0.70 = 0.126.
\]

With 850 observations, its expected count is:

\[
850 \times 0.126 = 107.1.
\]

The actual count will vary from one generated dataset to another.


In [4]:
frequency = (
    observations["sentence"]
    .value_counts()
    .rename_axis("sentence")
    .reset_index(name="count")
)

frequency["observed_probability"] = frequency["count"] / N_OBSERVATIONS

frequency


,sentence,count,observed_probability
0,s r r r r f e,126,0.148235
1,s l l f e,114,0.134118
2,s f e,107,0.125882
3,s l l r r f e,105,0.123529
4,s r r f e,94,0.110588
5,i l l l l,50,0.058824
6,i l l,37,0.043529
7,s p e,31,0.036471
8,s l l p e,31,0.036471
9,s l l r r p e,29,0.034118


In [5]:
# Compare observed probabilities with the theoretical probabilities
# obtained directly from the grammar.

theoretical = {
    "s f e": 0.90 * 0.20 * 0.70,
    "s p e": 0.90 * 0.20 * 0.20,
    "s e": 0.90 * 0.20 * 0.10,

    "s r r f e": 0.90 * 0.20 * 0.70,
    "s r r p e": 0.90 * 0.20 * 0.20,
    "s r r e": 0.90 * 0.20 * 0.10,

    "s r r r r f e": 0.90 * 0.20 * 0.70,
    "s r r r r p e": 0.90 * 0.20 * 0.20,
    "s r r r r e": 0.90 * 0.20 * 0.10,

    "s l l f e": 0.90 * 0.20 * 0.70,
    "s l l p e": 0.90 * 0.20 * 0.20,
    "s l l e": 0.90 * 0.20 * 0.10,

    "s l l r r f e": 0.90 * 0.20 * 0.70,
    "s l l r r p e": 0.90 * 0.20 * 0.20,
    "s l l r r e": 0.90 * 0.20 * 0.10,

    "i l l": 0.10 * 0.50,
    "i l l l l": 0.10 * 0.50,
}

comparison = pd.DataFrame({
    "theoretical_probability": pd.Series(theoretical),
    "observed_count": observations["sentence"].value_counts(),
})
comparison["observed_probability"] = comparison["observed_count"] / N_OBSERVATIONS

comparison.sort_index()


,theoretical_probability,observed_count,observed_probability
i l l,0.050,37,0.043529
i l l l l,0.050,50,0.058824
s e,0.018,15,0.017647
s f e,0.126,107,0.125882
s l l e,0.018,10,0.011765
s l l f e,0.126,114,0.134118
s l l p e,0.036,31,0.036471
s l l r r e,0.018,17,0.020000
s l l r r f e,0.126,105,0.123529
s l l r r p e,0.036,29,0.034118
